# Customer Support Ticket Router with LangGraph

This notebook routes customer support tickets to billing or technical support using simple keyword rules. No LLM or API key is required. Run the code cells from top to bottom in Google Colab.

## 1. Install LangGraph
Install the library used to define the shared state, processing nodes, and routing edges. Run this setup cell when starting a fresh Colab runtime.

In [ ]:
# Run this setup cell first in Google Colab. No API key is needed.
%pip install -q "langgraph>=1.2.20,<2" "ipywidgets>=8.1.9,<9"


## 2. Define the State, Nodes, and Conditional Routing

`TicketState` stores the original `ticket`, its `category`, and the generated `reply`. Nodes return only the fields they change; LangGraph merges those updates into the state.

The `classify` node checks the lowercase ticket for billing keywords: `invoice`, `payment`, `refund`, `charged`, or `billing`. A matching substring selects billing; all other tickets go to technical support. This is a simple demonstration rule, so ambiguous tickets may be misclassified.

After classification, the conditional edge calls `route_ticket` to select either the `billing` node or the `technical` node. Each node supplies a fixed reply, then ends the workflow. `compile()` prepares the graph for execution.

**Flow:** Start → Classify → Billing or Technical → End.

In [ ]:
from typing import Literal, TypedDict

from langgraph.graph import END, START, StateGraph


# Shared state: each node reads the current values and returns updates.
class TicketState(TypedDict):
    ticket: str
    category: str
    reply: str


def classify(state: TicketState) -> dict:
    # Billing keywords take priority; other tickets go to technical support.
    billing_keywords = ('invoice', 'payment', 'refund', 'charged', 'billing')
    ticket = state['ticket'].lower()
    category = 'billing' if any(word in ticket for word in billing_keywords) else 'technical'
    return {'category': category}


# The conditional edge uses this function to choose the next node.
def route_ticket(state: TicketState) -> Literal['billing', 'technical']:
    return 'billing' if state['category'] == 'billing' else 'technical'


def billing(state: TicketState) -> dict:
    return {'reply': 'Our billing team can help. Please provide your invoice number.'}


def technical(state: TicketState) -> dict:
    return {'reply': 'Our technical team can help. Please share the error message and steps to reproduce it.'}


# Connect the nodes and compile the definition into an executable graph.
builder = StateGraph(TicketState)
builder.add_node('classify', classify)
builder.add_node('billing', billing)
builder.add_node('technical', technical)
builder.add_edge(START, 'classify')
builder.add_conditional_edges(
    'classify',
    route_ticket,
    {'billing': 'billing', 'technical': 'technical'},
)
builder.add_edge('billing', END)
builder.add_edge('technical', END)
graph = builder.compile()


## 3. Try One Ticket for Each Route

The first ticket contains `charged` and `invoice`, so it should reach billing. The second describes an app crash and has no billing keywords, so it should reach technical support.

Each `graph.invoke()` call starts with a new state and returns the completed state. The cell prints the original ticket, selected category, and support reply. Change the sample text to explore the keyword rules.

In [ ]:
# Run one sample ticket through each route.
sample_tickets = [
    'I was charged twice for my latest invoice.',
    'The app crashes whenever I try to sign in.',
]

for ticket in sample_tickets:
    result = graph.invoke({'ticket': ticket, 'category': '', 'reply': ''})
    print('Ticket:', result['ticket'])
    print('Category:', result['category'])
    print('Reply:', result['reply'])
    print()


## 4. Check Routing and Edge Cases

These assertions check both routes, uppercase text, mixed topics, multiline input, Unicode, and the existing substring matching behavior. Billing keywords keep priority; tickets without them go to Technical. These checks use the real compiled graph, with no LLM or credentials.


In [ ]:
routing_cases = [
    ('I was charged twice.', 'billing'),
    ('Please send my invoice.', 'billing'),
    ('Can I get a REFUND?', 'billing'),
    ('My payment failed.', 'billing'),
    ('I have a billing question.', 'billing'),
    ('The app crashes on login.', 'technical'),
    ('My password reset link is broken.', 'technical'),
    ('My connection keeps timing out.', 'technical'),
    ('The app crashes and I need a refund.', 'billing'),
    ('   INVOICE question   ', 'billing'),
    ('First line\nPayment question on the next line.', 'billing'),
    ('The screen is frozen 😟', 'technical'),
    ('Hello!', 'technical'),
    ('', 'technical'),
    ('   ', 'technical'),
    ('The prepayment page is stuck.', 'billing'),
]

for ticket, expected_category in routing_cases:
    result = graph.invoke({'ticket': ticket, 'category': '', 'reply': ''})
    assert result['category'] == expected_category, repr(ticket)
    assert result['ticket'] == ticket
    expected_reply = billing(result)['reply'] if expected_category == 'billing' else technical(result)['reply']
    assert result['reply'] == expected_reply, repr(ticket)

print(f'Passed {len(routing_cases)} routing checks.')
# The graph retains its original fallback for empty text; the interface rejects it.


## 5. Submit Your Own Tickets

Run the next cell once, type a customer support ticket, and click **Submit ticket**. The result shows **Classification: Billing or Technical** and **Response**. Edit the text and submit again as often as you like while the Colab runtime remains connected. The latest result replaces the previous result; **Clear** resets the text and result.

Empty or whitespace-only submissions show a helpful message. Ticket text is displayed as plain escaped content. The interface uses the same compiled graph and keyword rules as the examples above, with no API key. Rerunning this cell replaces the old controls.


In [ ]:
from html import escape

import ipywidgets as widgets
from IPython.display import display


class TicketRouterInterface:
    def __init__(self, ticket_graph):
        self.graph = ticket_graph
        self.ticket_input = widgets.Textarea(
            placeholder='Type a customer support ticket here...',
            description='Ticket:',
            layout=widgets.Layout(width='100%', height='120px'),
        )
        self.submit_button = widgets.Button(
            description='Submit ticket', button_style='primary', icon='paper-plane'
        )
        self.clear_button = widgets.Button(description='Clear', icon='eraser')
        self.result = widgets.HTML(value='')
        self.submit_button.on_click(self.submit)
        self.clear_button.on_click(self.clear)
        self.view = widgets.VBox([
            widgets.HTML('<h3>Test a Customer Support Ticket</h3>'),
            self.ticket_input,
            widgets.HBox([self.submit_button, self.clear_button]),
            self.result,
        ], layout=widgets.Layout(width='100%', max_width='850px'))

    def submit(self, _button=None):
        ticket = self.ticket_input.value.strip()
        if not ticket:
            self.result.value = '<p role="alert">Please enter a ticket before submitting.</p>'
            return

        self.submit_button.disabled = True
        self.result.value = '<p role="status">Classifying ticket...</p>'
        try:
            state = self.graph.invoke({'ticket': ticket, 'category': '', 'reply': ''})
            category = escape(state['category'].title())
            reply = escape(state['reply'])
            safe_ticket = escape(ticket)
            self.result.value = (
                '<div role="status" style="border:1px solid #888;padding:16px;'
                'margin-top:12px;overflow-wrap:anywhere">'
                f'<p><strong>Classification:</strong> {category}</p>'
                f'<p><strong>Response:</strong> {reply}</p>'
                f'<p style="white-space:pre-wrap"><strong>Ticket:</strong> {safe_ticket}</p>'
                '</div>'
            )
        except Exception:
            # Keep controls usable without displaying internal error details.
            self.result.value = (
                '<p role="alert">Unable to process this ticket. '
                'Run the graph setup cell again, then retry.</p>'
            )
        finally:
            self.submit_button.disabled = False

    def clear(self, _button=None):
        self.ticket_input.value = ''
        self.result.value = ''

    def close(self):
        self.submit_button.on_click(self.submit, remove=True)
        self.clear_button.on_click(self.clear, remove=True)
        for child in self.view.children:
            child.close()
        self.ticket_input.close()
        self.submit_button.close()
        self.clear_button.close()
        self.result.close()
        self.view.close()


# Dispose of previous controls if this cell is run again.
if 'ticket_router_ui' in globals():
    ticket_router_ui.close()
ticket_router_ui = TicketRouterInterface(graph)
display(ticket_router_ui.view)


## 6. Draw the Graph

Display the compiled graph as an image to see the classification step, both conditional routes, and the end of each path. The diagram shows the possible paths, rather than the execution trace of a particular ticket.

Image rendering uses an online Mermaid service and requires internet access. Only the graph structure is sent for rendering; ticket contents are not included.

If the image service is unavailable, the cell displays the Mermaid diagram source instead. The interactive router remains usable.


In [ ]:
from IPython.display import Image, display

# Rendering uses an online Mermaid service; tickets are not sent to it.
try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    # A renderer outage should not prevent using the ticket interface.
    print('Graph image unavailable. The ticket router still works.')
    print(graph.get_graph().draw_mermaid())
